# Logical CNOT by lattice surgery

*A learning-tool walkthrough — surface-code series, notebook 5.*

The earlier notebooks built a logical qubit, a decoder, two entangled qubits with
transversal Clifford gates, and universal computation via $T$-gate teleportation
from magic states. One thing kept breaking: a **mid-circuit logical $H$ followed by
a transversal CNOT**. A
transversal CNOT only implements a *logical* CNOT when the two patches' $X$/$Z$
check supports line up qubit-for-qubit; a logical $H$ leaves a patch in the dual
frame, so the alignment fails and `CNOT·(H⊗I) ∼ CZ`, which kicks the state out of
the code space.

The honest fix used by fixed-layout hardware (Google/IBM) is **lattice surgery**:
never do a transversal two-qubit gate — realise the logical CNOT by **measuring
joint logical operators** (Horsman *et al.* 2012; Litinski 2019). Because the CNOT
is now a *measurement* pattern plus classical corrections, it composes correctly
with a Hadamard frame.

**What this notebook does**

1. Derives the lattice-surgery CNOT — ancilla $|+\rangle_L$, a $ZZ$-merge, an
   $XX$-merge, an ancilla read-out — and the **classical byproduct corrections**.
2. Verifies it on **3 bare qubits** (transparent state-vector), where it equals
   CNOT on every input.
3. Realises the joint measurements on the **rotated-planar $d{=}3$ patches** using
   the project's ITensor conventions, and **verifies the headline case
   $H_1$ then $\mathrm{CNOT}_{12}$ builds a Bell state** — the composition the
   transversal CNOT could not do.

## 0 · Conventions (unchanged from the earlier notebooks)

Everything is the same rotated-planar $d=3$ encoding as before, bundled into
`surface_code_ls.jl`:

* **3 patches, side-by-side on one MPS** — all of patch 1, then patch 2, then
  patch 3 (data, then $Z$-aux, then $X$-aux within each). Side-by-side (not
  interleaved) so a between-patch operation's non-locality cost stays visible.
* Patches **1, 2** are the logical data qubits; **patch 3** is a recyclable
  ancilla (magic-state injection *and* the lattice-surgery ancilla).
* Logical $Z = Z$ on **row $y=0$**; logical $X = X$ on **column $x=0$**.
* **H-parity frame.** A transversal logical $H$ sends a patch to its dual code:
  each logical operator keeps its Pauli *type* but its support swaps
  row$0\leftrightarrow$col$0$. We track this with a flag `hpar` and the helper
  `logop(L, hpar)`, so every logical operation stays frame-aware.

In [ ]:
include("functions/surface_code_ls.jl")       # base machinery + conventions
include("functions/lattice_surgery_ops.jl")   # the lattice-surgery CNOT (defined below)
println("loaded: ", length(all_keys), " sites  (3 patches × ",
        length(data_coords), " data + ", Nz_per_patch, " Z-aux + ",
        length(x_aux_local), " X-aux)")

## 1 · The idea: a CNOT built from joint measurements

Lattice surgery **measures joint logical operators**. Merging two patches along a
shared boundary and reading the new boundary stabilizers measures either
$Z_L\!\otimes\! Z_L$ (a *smooth*/$ZZ$ merge) or $X_L\!\otimes\! X_L$ (a *rough*/$XX$
merge) of the two patches — **without** revealing either patch's logical value on
its own. That single joint parity is the primitive we build the CNOT from.

**The gadget** (Horsman *et al.*; Litinski). With control $c$, target $t$, and a
fresh **ancilla $a$ prepared in $|+\rangle_L$**:

$$
\underbrace{M_{ZZ}(c,a)}_{m_1}\;\longrightarrow\;
\underbrace{M_{XX}(a,t)}_{m_2}\;\longrightarrow\;
\underbrace{M_{Z}(a)}_{m_3}
$$

i.e. a $ZZ$-merge of control with ancilla, an $XX$-merge of ancilla with target,
then destructively read the ancilla out in $Z$. The three outcomes $m_1,m_2,m_3$
determine a Pauli **byproduct** that we track classically (next section).

> **On "measure $Z_cZ_t$ then $X_cX_t$."** The two joint measurements are between
> *control–ancilla* and *ancilla–target*, not directly between control and target.
> Measuring $Z_cZ_t$ and then $X_cX_t$ *directly* on the two data patches would
> project them into a Bell basis and **erase the input** — those two operators
> anticommute in a way that destroys the data. The ancilla in $|+\rangle_L$ is the
> standard device that turns two joint parity measurements into a genuine unitary
> CNOT. It is exactly the role the shared boundary/ancilla patch plays in a real
> merge–split.

## 2 · The classical corrections (stabilizer flow)

Track how the input logical operators propagate through the three measurements.
Write $s_i=(-1)^{m_i}$; the ancilla starts stabilized by $+X_a$.

| input operator | commutes with $M_{ZZ},M_{XX},M_Z$? | result |
|---|---|---|
| $Z_c$ | yes, yes, yes | $Z_c^{\text{out}} = Z_c^{\text{in}}$ |
| $X_t$ | yes, yes, yes | $X_t^{\text{out}} = X_t^{\text{in}}$ |
| $Z_t$ | anticommutes with $M_{XX}$ | fix with $M_{ZZ}$, then $Z_a\!\to\! s_3$: $\;Z_c Z_t^{\text{out}} = s_1 s_3\, Z_t^{\text{in}}$ |
| $X_c$ | anticommutes with $M_{ZZ}$ | fix with $M_{XX}$, then $Z_a\!\to\! s_3$: $\;X_c X_t^{\text{out}} = s_2\, X_c^{\text{in}}$ |

The first two rows are CNOT's $Z_c\!\to\!Z_c$ and $X_t\!\to\!X_t$; the last two are
CNOT's $Z_t\!\to\!Z_cZ_t$ and $X_c\!\to\!X_cX_t$ — **up to the signs** $s_1s_3$ and
$s_2$. So the gadget is CNOT followed by a Pauli byproduct. Undo the signs with

$$
\boxed{\;X_{\text{target}}^{\,m_1\oplus m_3}\quad\text{and}\quad Z_{\text{control}}^{\,m_2}\;}
$$

$X_t$ anticommutes with $Z_t$ (cancels the $s_1s_3=(-1)^{m_1\oplus m_3}$ sign on
$Z_cZ_t$); $Z_c$ anticommutes with $X_c$ (cancels the $s_2=(-1)^{m_2}$ sign on
$X_cX_t$). The two byproducts act on disjoint sectors, so they never interfere.

## 3 · Sanity check on 3 bare qubits

Before touching the surface code, run the exact protocol on **three ordinary
qubits** as a pure state vector — control $c$, target $t$, ancilla $a$ — where we
can compare the output directly to `CNOT|ψ⟩` on random inputs. A projective
measurement of a Pauli involution $P$ is just $\psi \to (\mathbb{1}\pm P)\psi/\lVert\cdot\rVert$
with outcome probability $\tfrac12(1\pm\langle\psi|P|\psi\rangle)$.

In [ ]:
using LinearAlgebra, Statistics, Random

const I2 = ComplexF64[1 0;0 1]; const Xg = ComplexF64[0 1;1 0]; const Zg = ComplexF64[1 0;0 -1]
kron3(a,b,c) = kron(kron(a,b),c)                       # qubit order (c, t, a)
const ZcZa = kron3(Zg,I2,Zg); const XaXt = kron3(I2,Xg,Xg); const Za = kron3(I2,I2,Zg)
const Xt_op = kron3(I2,Xg,I2); const Zc_op = kron3(Zg,I2,I2)
const CNOT_ct = ComplexF64[1 0 0 0;0 1 0 0;0 0 0 1;0 0 1 0]   # control c, target t

# projective measurement of a Hermitian Pauli involution P; returns (bit, ψ')
function measure_pauli(ψ, P)
    Id8 = Matrix{ComplexF64}(I,8,8)
    pplus = real(ψ' * (Id8 + P)/2 * ψ)
    if rand() < pplus
        φ = (Id8 + P)/2 * ψ; return 0, φ/norm(φ)
    else
        φ = (Id8 - P)/2 * ψ; return 1, φ/norm(φ)
    end
end

function bare_ls_cnot(ψct; correct=true)               # ψct :: 4-vector on (c,t)
    ψ = kron(ψct, ComplexF64[1,1]/√2)                  # ⊗ |+>_a
    m1, ψ = measure_pauli(ψ, ZcZa)                     # M_ZZ(control, ancilla)
    m2, ψ = measure_pauli(ψ, XaXt)                     # M_XX(ancilla, target)
    m3, ψ = measure_pauli(ψ, Za)                       # read ancilla in Z
    if correct
        (m1 ⊻ m3) == 1 && (ψ = Xt_op*ψ)                # byproduct X_t^{m1⊕m3}
        m2 == 1        && (ψ = Zc_op*ψ)                # byproduct Z_c^{m2}
    end
    col = reshape(ψ, 2, 4)[m3+1, :]                    # ancilla collapsed to |m3>
    col / norm(col), (m1,m2,m3)
end

Random.seed!(1)
fids = Float64[]; raw = Float64[]
for _ in 1:2000
    v = randn(ComplexF64,4); v ./= norm(v)
    out,_ = bare_ls_cnot(v);            push!(fids, abs2((CNOT_ct*v)' * out))
    rw ,_ = bare_ls_cnot(v; correct=false); push!(raw, abs2((CNOT_ct*v)' * rw))
end
println("fidelity to CNOT|ψ⟩ over 2000 random inputs:  min=", round(minimum(fids),digits=6),
        "  mean=", round(mean(fids),digits=6))
println("same, WITHOUT the classical corrections:      mean=", round(mean(raw),digits=3),
        "  (byproducts matter!)")

With the byproducts applied, the gadget reproduces CNOT **exactly** (fidelity
$1$) on every random input; drop the corrections and it is only right by chance.
That is the whole protocol — now we implement the two joint measurements on real
surface-code patches.

## 4 · Joint measurements on the surface code

Physically, $M_{ZZ}$/$M_{XX}$ are **merge–split** operations: temporarily stitch two
patches together along a boundary, measure the new boundary stabilizers ($d$ rounds
for fault tolerance), and split them apart; the product of the seam outcomes is the
joint logical parity. In this state-vector model we realise the *same projective
measurement* directly and exactly. Because a logical operator
$O = O_c \otimes O_a$ (a product of physical Paulis on the two logical supports)
commutes with every stabilizer, projecting onto its $\pm1$ eigenspace keeps both
patches in the code space and only fixes their joint parity:

$$
\text{outcome }\pm\text{ with prob }\tfrac12\bigl(1\pm\langle\psi|O|\psi\rangle\bigr),
\qquad \psi \;\to\; (\mathbb{1}\pm O)\,\psi \,/\,\lVert\cdot\rVert .
$$

This is `measure_joint_exact!` in `lattice_surgery_ops.jl`. Doing the projection
directly — rather than threading a parity ancilla through ~35 sites with a long
non-local controlled-gate circuit — avoids the truncation error such a circuit
accumulates in the MPS. `logop(L, hpar)` supplies the correct support for each
patch, so the gadget is **frame-aware** and composes with a logical $H$.

```julia
# EXACT projective joint logical-Pauli measurement; specs = [(patch, L, hpar), ...]
function measure_joint_exact!(psi, specs; cutoff = threshold)
    Opsi = _apply_joint(psi, specs; cutoff)          # O = ∏ physical Paulis on the supports
    ex = real(inner(psi, Opsi))                      # <ψ|O|ψ>
    if rand() < 0.5*(1 + ex)
        bit = 0; newpsi = +(psi,  Opsi; cutoff)      # ∝ (1+O)ψ
    else
        bit = 1; newpsi = +(psi, -Opsi; cutoff)      # ∝ (1-O)ψ
    end
    bit, newpsi / sqrt(real(inner(newpsi, newpsi)))
end

# the gadget: ancilla|+>, ZZ-merge, XX-merge, read ancilla, apply byproducts
function ls_cnot!(psi, pc, pt; hc=0, ht=0, cutoff = threshold)
    psi = inject_det!(psi, 3, :plus; cutoff)                             # ancilla |+>_L
    m1, psi = measure_joint_exact!(psi, [(pc,:Z,hc),(3,:Z,0)]; cutoff)   # M_ZZ(control,ancilla)
    m2, psi = measure_joint_exact!(psi, [(3,:X,0),(pt,:X,ht)]; cutoff)   # M_XX(ancilla,target)
    m3, psi = measure_joint_exact!(psi, [(3,:Z,0)]; cutoff)              # M_Z(ancilla)
    (m1 ⊻ m3) == 1 && (psi = applyL!(psi, pt, ht, :X; cutoff))           # X_target^{m1⊕m3}
    m2 == 1        && (psi = applyL!(psi, pc, hc, :Z; cutoff))           # Z_control^{m2}
    psi, (m1, m2, m3)
end
```

We prepare logical states deterministically (`inject_det!`: seed the corner, then
force every stabilizer to $+1$) so the reference is unambiguous, and verify against
an exact 2-qubit state-vector.

In [ ]:
# exact classical 2-qubit reference (same one used by the circuit runner)
const _ket = Dict(:zero=>ComplexF64[1,0], :one=>ComplexF64[0,1], :plus=>ComplexF64[1,1]/√2,
                  :minus=>ComplexF64[1,-1]/√2, :A=>ComplexF64[1,exp(im*π/4)]/√2, :Y=>ComplexF64[1,im]/√2)
const _Hc=ComplexF64[1 1;1 -1]/√2; const _Sc=ComplexF64[1 0;0 im]; const _Tc=ComplexF64[1 0;0 exp(im*π/4)]
const _Xc=ComplexF64[0 1;1 0]; const _Zc=ComplexF64[1 0;0 -1]; const _Ic=ComplexF64[1 0;0 1]
_1q(g,p) = p==1 ? kron(g,_Ic) : kron(_Ic,g)
const _CNOT12 = ComplexF64[1 0 0 0;0 1 0 0;0 0 0 1;0 0 1 0]
function ref_state(circuit; inits=(:zero,:zero))
    ψ = kron(_ket[inits[1]], _ket[inits[2]])
    for o in circuit
        g=o[1]
        g==:H ? (ψ=_1q(_Hc,o[2])*ψ) : g==:S ? (ψ=_1q(_Sc,o[2])*ψ) : g==:T ? (ψ=_1q(_Tc,o[2])*ψ) :
        g==:X ? (ψ=_1q(_Xc,o[2])*ψ) : g==:Z ? (ψ=_1q(_Zc,o[2])*ψ) : g==:CNOT ? (ψ=_CNOT12*ψ) : error("op")
    end
    ψ
end
const _Pauli = Dict(:X=>_Xc, :Y=>ComplexF64[0 -im;im 0], :Z=>_Zc, :I=>_Ic)
ref_corr(ψ,b1,b2) = real(ψ' * kron(_Pauli[b1],_Pauli[b2]) * ψ)

Random.seed!(0xBE11)
println("sanity · deterministic injection reads out ±1 as intended:")
for (s,L,want) in [(:zero,:Z,"+1"),(:one,:Z,"-1"),(:plus,:X,"+1"),(:minus,:X,"-1")]
    vals=[ev(readout((psi=MPS(sites,"Up"); inject_det!(psi,1,s)),1,0,L)) for _ in 1:8]
    println("  inject $s read $L:  mean=", round(mean(vals),digits=2), "  (want $want)")
end

### Verify the LS-CNOT against the exact CNOT on every basis input

For each product input we run the gadget, then compare the **corrected** joint
expectations $\langle Z_1Z_2\rangle,\langle X_1X_2\rangle$ to `CNOT` on the exact
reference. (We read joint expectations directly; the subtlety of *sampled* joint
read-out is covered just below.)

In [ ]:
prep2(s1,s2) = (psi=MPS(sites,"Up"); psi=inject_det!(psi,1,s1); psi=inject_det!(psi,2,s2); psi)
function corr_expects(prep; hc=0, ht=0, n=12)
    zz=Float64[]; xx=Float64[]
    for _ in 1:n
        psi,_ = ls_cnot!(prep(), 1, 2; hc=hc, ht=ht)
        push!(zz, joint_expect(psi,[(1,:Z,hc),(2,:Z,ht)]))
        push!(xx, joint_expect(psi,[(1,:X,hc),(2,:X,ht)]))
    end
    mean(zz), mean(xx)
end

println("surface-code LS-CNOT(1→2)  vs  exact CNOT reference")
for (s1,s2) in [(:zero,:zero),(:one,:zero),(:plus,:zero),(:minus,:zero),(:plus,:plus),(:one,:one)]
    zz,xx = corr_expects(()->prep2(s1,s2))
    ψ = ref_state([(:CNOT,1,2)]; inits=(s1,s2))
    println("  ($s1,$s2):  ⟨Z₁Z₂⟩ sim=", lpad(round(zz,digits=2),5), " ref=", lpad(round(ref_corr(ψ,:Z,:Z),digits=2),5),
            "  |  ⟨X₁X₂⟩ sim=", lpad(round(xx,digits=2),5), " ref=", lpad(round(ref_corr(ψ,:X,:X),digits=2),5))
end

Every row matches the exact CNOT — including the genuinely **entangling**
`(plus,zero)` $\to$ Bell $\tfrac{1}{\sqrt2}(|00\rangle+|11\rangle)$ (both
correlations $+1$) and `(minus,zero)` $\to \tfrac{1}{\sqrt2}(|00\rangle-|11\rangle)$
($\langle Z_1Z_2\rangle=+1,\ \langle X_1X_2\rangle=-1$).

## 5 · Headline — $H_1$ then $\mathrm{CNOT}_{12}$ builds a Bell state

This is the case the transversal CNOT could not do. We apply a **real transversal
logical $H$** to patch 1 (flipping its H-parity to `hpar=1`), then the frame-aware
LS-CNOT. The result should be the Bell state
$\mathrm{CNOT}_{12}\,(H\otimes I)|00\rangle = \tfrac{1}{\sqrt2}(|00\rangle+|11\rangle)$:
$\langle Z_1Z_2\rangle=\langle X_1X_2\rangle=+1$, with each single-qubit expectation
$0$.

In [ ]:
zz=Float64[]; xx=Float64[]; z1=Float64[]; x1=Float64[]
for _ in 1:16
    psi = MPS(sites,"Up"); psi = inject_det!(psi,1,:zero); psi = inject_det!(psi,2,:zero)
    psi = tH!(psi,1)                              # real transversal logical H on control → hpar 1
    psi,_ = ls_cnot!(psi, 1, 2; hc=1, ht=0)       # frame-aware LS-CNOT
    push!(zz, joint_expect(psi,[(1,:Z,1),(2,:Z,0)])); push!(xx, joint_expect(psi,[(1,:X,1),(2,:X,0)]))
    push!(z1, joint_expect(psi,[(1,:Z,1)]));          push!(x1, joint_expect(psi,[(1,:X,1)]))
end
ψ = ref_state([(:H,1),(:CNOT,1,2)]; inits=(:zero,:zero))
println("H₁ then LS-CNOT(1→2):")
println("  ⟨Z₁Z₂⟩ sim=", round(mean(zz),digits=2), " ref=", round(ref_corr(ψ,:Z,:Z),digits=2),
        "   ⟨X₁X₂⟩ sim=", round(mean(xx),digits=2), " ref=", round(ref_corr(ψ,:X,:X),digits=2))
println("  ⟨Z₁⟩ sim=", round(mean(z1),digits=2), "  ⟨X₁⟩ sim=", round(mean(x1),digits=2), "  (Bell ⇒ 0)")

### A read-out pitfall worth knowing

To *verify* a joint parity you must measure both patches on the **same collapsing
state** (or read the joint operator's expectation, as above). Measuring each patch
on an **independent copy** — e.g. `ev(readout(psi,1))·ev(readout(psi,2))`, where
`readout` internally copies `psi` — destroys the correlation of an entangled state:
each patch's individual logical value is random, so the product looks like noise
even when the Bell state is perfect. `joint_readout!` does it correctly on one
state. (This artifact is exactly why an earlier investigation *appeared* to show the
gadget failing on entangling inputs — the gadget was fine; the check was not.)

### References
- D. Horsman, A. G. Fowler, S. Devitt, R. Van Meter — *Surface code quantum computing by lattice surgery* (2012).
- D. Litinski — *A Game of Surface Codes* (2019).
- C. Gidney, A. G. Fowler — lattice-surgery constructions and twist-free variants.